<a href="https://colab.research.google.com/github/jonesrmj/MiDuRy-SentiBal/blob/mizuho/MiDuRy-SentiBal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

mizuho branch
- https://colab.research.google.com/github/jonesrmj/MiDuRy-SentiBal/blob/mizuho/MiDuRy-SentiBal.ipynb

# Setup

In [7]:
import os

# Clone repo if running in Colab
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('MiDuRy-SentiBal'):
        !git clone https://github.com/jonesrmj/MiDuRy-SentiBal.git
    os.chdir('MiDuRy-SentiBal')

import numpy as np
import pandas as pd

print("Working directory:", os.getcwd())

Cloning into 'MiDuRy-SentiBal'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 50 (delta 14), reused 30 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 5.67 MiB | 9.61 MiB/s, done.
Resolving deltas: 100% (14/14), done.
Working directory: /content/MiDuRy-SentiBal/MiDuRy-SentiBal


In [8]:
import os

# Mount Google Drive (Colab only) for saving model outputs
if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_OUTPUT_PATH = '/content/drive/MyDrive/CISC484_S26/Training_Results'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Data Loading

In [9]:
df = pd.read_csv('data/tweets.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

Shape: (56221, 5)
Columns: ['Unnamed: 0', 'Datetime', 'Tweet Id', 'Text', 'Username']


,Unnamed: 0,Datetime,Tweet Id,Text,Username
0,0,2023-04-19 21:27:19+00:00,1648800467206672384,From Studio Gangster to Synthetic Gangster 🎤.....,resembleai
1,1,2023-04-19 21:27:09+00:00,1648800425540476929,Took me some time to find this. I build this #...,devaanparbhoo
2,2,2023-04-19 21:26:57+00:00,1648800376479715328,Mind blowing next wave #generativeai platform...,timreha
3,3,2023-04-19 21:26:49+00:00,1648800341193027584,Open Source Generative AI Image Specialist Sta...,VirtReview
4,4,2023-04-19 21:25:00+00:00,1648799883934203905,Are you an #HR leader considering which future...,FrozeElle


# Preprocessing Pipeline
Lowercase text, remove URLs, mentions, punctuation, numbers, special characters, extra whitespace, stop words, and hashtag symbols (retaining the words). Store result in `cleaned_text` column.

In [10]:
import re
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

def clean_text(text):
    """Clean a single tweet text string."""
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text) # Remove URLs
    text = re.sub(r'@\w+', '', text) # Remove mentions
    text = re.sub(r'#(\w+)', r'\1', text) # Remove # but keep the word
    text = re.sub(r'[^\w\s]', '', text) # Remove punctuation & special chars
    text = re.sub(r'\d+', '', text) # Remove numbers
    text = re.sub(r'\s+', ' ', text).strip() # Collapse extra whitespace
    text = ' '.join(w for w in text.split() if w not in stop_words)
    return text

df['cleaned_text'] = df['Text'].astype(str).apply(clean_text)

print(f"Rows: {len(df)}")
print(f"Sample cleaned texts:")
df[['Text', 'cleaned_text']].head(10)

Rows: 56221
Sample cleaned texts:


,Text,cleaned_text
0,From Studio Gangster to Synthetic Gangster 🎤.....,studio gangster synthetic gangster investigate...
1,Took me some time to find this. I build this #...,took time find build nocode prototype dec real...
2,Mind blowing next wave #generativeai platform...,mind blowing next wave generativeai platform c...
3,Open Source Generative AI Image Specialist Sta...,open source generative ai image specialist sta...
4,Are you an #HR leader considering which future...,hr leader considering future trends prioritize...
5,#GenerativeAI is a new technology that can cre...,generativeai new technology create new text im...
6,Salesforce announces plans to integrate Einste...,salesforce announces plans integrate einstein ...
7,Discover the limitless possibilities of #Gener...,discover limitless possibilities generativeai ...
8,Local art fam: Come join us at Adobe SF to lea...,local art fam come join us adobe sf learn fire...
9,Salesforce announces plans to integrate Einste...,salesforce announces plans integrate einstein ...


# Sentiment Label Generation
generates labels with VADER

In [11]:
! pip install vaderSentiment

In [12]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_vader_label(text):
  score = analyzer.polarity_scores(text)
  compound = score['compound']

  if compound >= 0.05:
    return 'positive'
  elif compound <= -0.05:
    return 'negative'
  else:
    return 'neutral'

df['vader_label'] = df['cleaned_text'].apply(get_vader_label)
df.to_csv('data_with_vader_labels.csv', index=False)


In [13]:
df[['vader_label', 'cleaned_text']].head(10)

,vader_label,cleaned_text
0,negative,studio gangster synthetic gangster investigate...
1,neutral,took time find build nocode prototype dec real...
2,neutral,mind blowing next wave generativeai platform c...
3,neutral,open source generative ai image specialist sta...
4,neutral,hr leader considering future trends prioritize...
5,positive,generativeai new technology create new text im...
6,positive,salesforce announces plans integrate einstein ...
7,positive,discover limitless possibilities generativeai ...
8,positive,local art fam come join us adobe sf learn fire...
9,positive,salesforce announces plans integrate einstein ...


## Apply TF-IDF


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse
import joblib

documents = df['cleaned_text']

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X = vectorizer.fit_transform(documents)

print(X.shape)

scipy.sparse.save_npz('tfidf_matrix.npz', X)
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

(56221, 5000)


['tfidf_vectorizer.pkl']